# Tennis Odds Preprocessing

In [5]:
# Libraries used for odds-data inspection and preprocessing

import polars as pl
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO

## 1. Setup and locate the raw ZIP data

In [10]:
# Locate the dataset from the preprocessing folder inside the Git repository

MINI_PROJECT_ROOT = Path.cwd().parents[2]

tennis_data = MINI_PROJECT_ROOT / "Tennis Schema"
data_folder = tennis_data / "tennis_data"

print(data_folder)
print(data_folder.exists())

d:\Learning\Daneshkar\Statistics\Stat with Python\Mini Project\Tennis Schema\tennis_data
True


In [11]:
# Inspect the contents of the first daily ZIP archive and understand how the raw folders and Parquet files are organized

first_zip = data_folder / "20240201.zip"

with ZipFile(first_zip) as z:
    files_inside = z.namelist()

files_inside[:20]

['../../data/raw/raw_match_parquet/away_team_11998445.parquet',
 '../../data/raw/raw_match_parquet/away_team_11998446.parquet',
 '../../data/raw/raw_match_parquet/away_team_11998447.parquet',
 '../../data/raw/raw_match_parquet/away_team_11998448.parquet',
 '../../data/raw/raw_match_parquet/away_team_11998449.parquet',
 '../../data/raw/raw_match_parquet/away_team_11998450.parquet',
 '../../data/raw/raw_match_parquet/away_team_11998451.parquet',
 '../../data/raw/raw_match_parquet/away_team_11998456.parquet',
 '../../data/raw/raw_match_parquet/away_team_11998459.parquet',
 '../../data/raw/raw_match_parquet/away_team_11998666.parquet',
 '../../data/raw/raw_match_parquet/away_team_11998667.parquet',
 '../../data/raw/raw_match_parquet/away_team_11998670.parquet',
 '../../data/raw/raw_match_parquet/away_team_11998671.parquet',
 '../../data/raw/raw_match_parquet/away_team_11998672.parquet',
 '../../data/raw/raw_match_parquet/away_team_11998674.parquet',
 '../../data/raw/raw_match_parquet/away_

In [12]:
# List the raw data categories assigned for preprocessing
# Count how many Parquet files from each category exist in the first daily ZIP

folders_to_clean = [
    "raw_odds_parquet",
    "raw_point_by_point_parquet",
    "raw_statistics_parquet",
    "raw_tennis_power_parquet",
    "raw_votes_parquet",
]

for folder in folders_to_clean:
    files = [
        f for f in files_inside
        if folder in f and f.endswith(".parquet")
    ]
    print(folder, len(files))

raw_odds_parquet 267
raw_point_by_point_parquet 265
raw_statistics_parquet 273
raw_tennis_power_parquet 271
raw_votes_parquet 285


## 2. Inspect the odds data structure

In [13]:
# Select only the odds Parquet files from the first daily ZIP
# Inspect one of them before processing the full odds dataset

odds_files = [
    f for f in files_inside
    if "raw_odds_parquet" in f and f.endswith(".parquet")
]

odds_files[0]

'../../data/raw/raw_odds_parquet/odds_11974053.parquet'

In [14]:
# Read the first odds Parquet file directly from inside the ZIP
# This gives a small sample to inspect before making any cleaning decisions

with ZipFile(first_zip) as z:
    with z.open(odds_files[0]) as f:
        odds_sample = pl.read_parquet(BytesIO(f.read()))

In [15]:
odds_sample.head()

match_id,market_id,market_name,is_live,suspended,initial_fractional_value,fractional_value,choice_name,choice_source_id,winnig,change
i64,i64,str,bool,bool,str,str,str,i64,bool,i64
11974053,1,"""full_time""",false,false,"""1/4""","""1/5""","""1""",1994690951,true,-1
11974053,1,"""full_time""",false,false,"""11/4""","""10/3""","""2""",1994691017,false,1


In [16]:
# Read a second odds Parquet file from the same daily ZIP
# Use a second sample to check whether the structure is consistent across different odds files before processing them together

second_odds_file = odds_files[1]

with ZipFile(first_zip) as z:
    with z.open(second_odds_file) as f:
        odds_sample_2 = pl.read_parquet(BytesIO(f.read()))

In [17]:
odds_sample_2

match_id,market_id,market_name,is_live,suspended,initial_fractional_value,fractional_value,choice_name,choice_source_id,winnig,change
i64,i64,str,bool,bool,str,str,str,i64,bool,i64
11974066,1,"""full_time""",false,false,"""14/1""","""14/1""","""1""",1990515041,false,0
11974066,1,"""full_time""",false,false,"""1/33""","""3/100""","""2""",1990515081,true,0


## 3. Investigate data quality issues

In [18]:
odds_sample.schema == odds_sample_2.schema

True

In [19]:
# Check all odds files from the first day for schema differences
# Identify columns whose data types may vary between files

different_schema = []

with ZipFile(first_zip) as z:
    for file in odds_files:
        with z.open(file) as f:
            schema = pl.read_parquet_schema(BytesIO(f.read())) #reads column names and data types, not the whole dataset

        if schema != odds_sample.schema:
            different_schema.append(file)

len(different_schema)

4

In [20]:
different_schema

['../../data/raw/raw_odds_parquet/odds_12011297.parquet',
 '../../data/raw/raw_odds_parquet/odds_12011306.parquet',
 '../../data/raw/raw_odds_parquet/odds_12021512.parquet',
 '../../data/raw/raw_odds_parquet/odds_12023361.parquet']

In [21]:
# Inspect one file with a different schema to determine which column caused the difference

with ZipFile(first_zip) as z:
    with z.open(different_schema[0]) as f:
        unusual_schema = pl.read_parquet_schema(BytesIO(f.read()))

unusual_schema

Schema([('match_id', Int64),
        ('market_id', Int64),
        ('market_name', String),
        ('is_live', Boolean),
        ('suspended', Boolean),
        ('initial_fractional_value', String),
        ('fractional_value', String),
        ('choice_name', String),
        ('choice_source_id', Int64),
        ('winnig', Null),
        ('change', Int64)])

The difference is that 'winning' is inferred as Null when all values in that column are missing

In [22]:
# Inspect the actual rows from a file where 'winnig' was inferred as Null

with ZipFile(first_zip) as z:
    with z.open(different_schema[0]) as f:
        unusual_odds = pl.read_parquet(BytesIO(f.read()))

unusual_odds

match_id,market_id,market_name,is_live,suspended,initial_fractional_value,fractional_value,choice_name,choice_source_id,winnig,change
i64,i64,str,bool,bool,str,str,str,i64,null,i64
12011297,1,"""full_time""",false,false,"""8/15""","""53/100""","""1""",1970744905,null,1
12011297,1,"""full_time""",false,false,"""11/8""","""69/50""","""2""",1970744920,null,-1
12011297,11,"""first_set_winner""",false,false,"""4/7""","""4/7""","""1""",1971636642,null,0
12011297,11,"""first_set_winner""",false,false,"""5/4""","""5/4""","""2""",1971636695,null,0


In [23]:
# Confirm that the schema differences are all caused by the 'winnig' column
# These files contain only null values, so Polars infers the column type as Null

with ZipFile(first_zip) as z:
    for file in different_schema:
        with z.open(file) as f:
            schema = pl.read_parquet_schema(BytesIO(f.read()))
        
        print(file)
        print(schema["winnig"])

../../data/raw/raw_odds_parquet/odds_12011297.parquet
Null
../../data/raw/raw_odds_parquet/odds_12011306.parquet
Null
../../data/raw/raw_odds_parquet/odds_12021512.parquet
Null
../../data/raw/raw_odds_parquet/odds_12023361.parquet
Null


In [24]:
# Combine all odds files from the first day
# Standardize 'winnig' as Boolean so files with all-null values can be concatenated safely

odds_dfs = []

with ZipFile(first_zip) as z:
    for file in odds_files:
        with z.open(file) as f:
            df = pl.read_parquet(BytesIO(f.read()))
        
        df = df.with_columns(
            pl.col("winnig").cast(pl.Boolean, strict=False)
        )
        
        odds_dfs.append(df)

odds_day1 = pl.concat(odds_dfs)

In [25]:
odds_day1.shape

(834, 11)

In [26]:
odds_day1.null_count()

match_id,market_id,market_name,is_live,suspended,initial_fractional_value,fractional_value,choice_name,choice_source_id,winnig,change
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,18,0


In [27]:
# Inspect rows where the winner information is missing

pl.Config.set_tbl_rows(20)
odds_day1.filter(
    pl.col("winnig").is_null()
)

match_id,market_id,market_name,is_live,suspended,initial_fractional_value,fractional_value,choice_name,choice_source_id,winnig,change
i64,i64,str,bool,bool,str,str,str,i64,bool,i64
12011297,1,"""full_time""",false,false,"""8/15""","""53/100""","""1""",1970744905,null,1
12011297,1,"""full_time""",false,false,"""11/8""","""69/50""","""2""",1970744920,null,-1
12011297,11,"""first_set_winner""",false,false,"""4/7""","""4/7""","""1""",1971636642,null,0
12011297,11,"""first_set_winner""",false,false,"""5/4""","""5/4""","""2""",1971636695,null,0
12011306,1,"""full_time""",false,false,"""4/5""","""4/5""","""1""",1970745950,null,0
12011306,1,"""full_time""",false,false,"""10/11""","""91/100""","""2""",1970746107,null,-1
12011306,11,"""first_set_winner""",false,false,"""4/5""","""4/5""","""1""",1971682556,null,0
12011306,11,"""first_set_winner""",false,false,"""10/11""","""10/11""","""2""",1971682629,null,0
12011306,12,"""total_games_won""",false,false,"""4/5""","""4/5""","""Over""",1971682549,null,0


In [28]:
# Count how many different matches contain missing winner information

odds_day1.filter(
    pl.col("winnig").is_null()
).select(
    pl.col("match_id").n_unique()
)

match_id
u32
4


In [29]:
# Check for exact duplicate rows
odds_day1.is_duplicated().sum()

0

In [30]:
# See what betting markets exist
odds_day1.group_by("market_name").len().sort("len", descending=True)

market_name,len
str,u32
"""full_time""",534
"""first_set_winner""",160
"""total_games_won""",140


In [31]:
# Check whether the odds strings are valid
odds_day1.select(
    pl.col("fractional_value").unique()
)

fractional_value
str
"""3/100"""
"""11/5"""
"""4/6"""
"""1/6"""
"""13/2"""
"""1/3"""
"""1/7"""
"""69/50"""
"""4/5"""


In [32]:
# Show any fractional_value that is not written as digits / digits
odds_day1.filter(
    ~pl.col("fractional_value").str.contains(r"^\d+/\d+$")
).select("fractional_value").unique()

fractional_value
str
"""4/"""


In [33]:
# Inspect the row and see whether initial_fractional_value gives us a clue
odds_day1.filter(
    pl.col("fractional_value") == "4/"
).select(
    "match_id",
    "market_name",
    "choice_name",
    "initial_fractional_value",
    "fractional_value",
    "winnig"
)

match_id,market_name,choice_name,initial_fractional_value,fractional_value,winnig
i64,str,str,str,str,bool
12024368,"""full_time""","""1""","""4/""","""4/""",true


In [34]:
# Checking the companion row to see the context
odds_day1.filter(
    pl.col("match_id") == 12024368
)

match_id,market_id,market_name,is_live,suspended,initial_fractional_value,fractional_value,choice_name,choice_source_id,winnig,change
i64,i64,str,bool,bool,str,str,str,i64,bool,i64
12024368,1,"""full_time""",false,false,"""4/""","""4/""","""1""",1994677274,true,0
12024368,1,"""full_time""",false,false,"""13/8""","""13/8""","""2""",1994677330,false,0


If a fractional-odds string is malformed, keep the row but replace that odds value with null rather than guessing.

In [35]:
# Check whether a betting selection appears more than once unexpectedly
odds_day1.group_by(
    ["match_id", "market_id", "choice_source_id"]
).len().filter(
    pl.col("len") > 1
)

match_id,market_id,choice_source_id,len
i64,i64,i64,u32


## 4. Investigate repeated daily snapshots

In [36]:
# Count all daily ZIP files
all_zips = sorted(data_folder.glob("*.zip"))

len(all_zips)

60

In [37]:
# How many odds Parquet files exist across all 60 days?
total_odds_files = 0

for zip_path in all_zips:
    with ZipFile(zip_path) as z:
        odds_files_in_zip = [
            f for f in z.namelist()
            if "raw_odds_parquet" in f and f.endswith(".parquet")
        ]

        total_odds_files += len(odds_files_in_zip)

total_odds_files

22065

In [38]:
# Check how many odds files are actually unique
all_odds_names = []

for zip_path in all_zips:
    with ZipFile(zip_path) as z:
        odds_files_in_zip = [
            f for f in z.namelist()
            if "raw_odds_parquet" in f and f.endswith(".parquet")
        ]

        all_odds_names.extend(odds_files_in_zip)

len(set(all_odds_names))

10476

In [39]:
# Find one repeated odds file
from collections import Counter

odds_name_counts = Counter(all_odds_names)

repeated_odds = [
    name for name, count in odds_name_counts.items()
    if count > 1
]

repeated_odds[0]

'../../data/raw/raw_odds_parquet/odds_11974053.parquet'

In [40]:
# Which daily ZIPs contain that same odds file
zips_with_repeat = []

for zip_path in all_zips:
    with ZipFile(zip_path) as z:
        if repeated_odds[0] in z.namelist():
            zips_with_repeat.append(zip_path.name)

zips_with_repeat

['20240201.zip', '20240202.zip', '20240203.zip']

In [41]:
# Compare the file contents
import hashlib

for zip_name in zips_with_repeat:
    zip_path = data_folder / zip_name

    with ZipFile(zip_path) as z:
        file_bytes = z.read(repeated_odds[0])

    file_hash = hashlib.md5(file_bytes).hexdigest()

    print(zip_name, file_hash)

20240201.zip 6ffdb695227c7cfa39bb40272ffaaead
20240202.zip 6ffdb695227c7cfa39bb40272ffaaead
20240203.zip 57284bb7db0b641138f5067ed3b5864d


Feb 1 and Feb 2 have identical Parquet files.  
Feb 3 has different Parquet file bytes, so we compare the actual DataFrames next to determine whether the underlying data changed.

In [42]:
# Compare Feb 1 vs Feb 3
versions = {}

for zip_name in ["20240201.zip", "20240203.zip"]:
    zip_path = data_folder / zip_name

    with ZipFile(zip_path) as z:
        file_bytes = z.read(repeated_odds[0])

    versions[zip_name] = pl.read_parquet(BytesIO(file_bytes))

In [43]:
versions["20240201.zip"]

match_id,market_id,market_name,is_live,suspended,initial_fractional_value,fractional_value,choice_name,choice_source_id,winnig,change
i64,i64,str,bool,bool,str,str,str,i64,bool,i64
11974053,1,"""full_time""",false,false,"""1/4""","""1/5""","""1""",1994690951,true,-1
11974053,1,"""full_time""",false,false,"""11/4""","""10/3""","""2""",1994691017,false,1


In [44]:
versions["20240203.zip"]

match_id,market_id,market_name,is_live,suspended,initial_fractional_value,fractional_value,choice_name,choice_source_id,winnig,change
i64,i64,str,bool,bool,str,str,str,i64,bool,i64
11974053,1,"""full_time""",false,false,"""1/4""","""1/5""","""1""",1994690951,true,-1
11974053,1,"""full_time""",false,false,"""11/4""","""10/3""","""2""",1994691017,false,1


In [45]:
# Compare the actual DataFrames
versions["20240201.zip"].equals(
    versions["20240203.zip"]
)

True

The odds data itself is identical, and the hash difference was only due to how the Parquet file was stored.

In [46]:
# Check every repeated odds file
changed_files = []

first_versions = {}

for zip_path in all_zips:
    with ZipFile(zip_path) as z:
        odds_files_in_zip = [
            f for f in z.namelist()
            if "raw_odds_parquet" in f and f.endswith(".parquet")
        ]

        for file in odds_files_in_zip:
            with z.open(file) as f:
                df = pl.read_parquet(BytesIO(f.read()))

            # Make the winnig type consistent
            df = df.with_columns(
                pl.col("winnig").cast(pl.Boolean, strict=False)
            )

            if file not in first_versions:
                first_versions[file] = df

            elif not df.equals(first_versions[file]):
                changed_files.append(file)

In [47]:
len(set(changed_files))

2072

In [48]:
unchanged_repeated = set(repeated_odds) - set(changed_files)

print("Repeated files:", len(repeated_odds))
print("Changed repeated files:", len(set(changed_files)))
print("Unchanged repeated files:", len(unchanged_repeated))

Repeated files: 10102
Changed repeated files: 2072
Unchanged repeated files: 8030


In [49]:
# Inspect one genuinely changed file
changed_file = list(set(changed_files))[0]

changed_file

'../../data/raw/raw_odds_parquet/odds_12111784.parquet'

In [50]:
# Which daily ZIPs contain this changed odds file
changed_file_zips = []

for zip_path in all_zips:
    with ZipFile(zip_path) as z:
        if changed_file in z.namelist():
            changed_file_zips.append(zip_path.name)

changed_file_zips

['20240228.zip', '20240229.zip', '20240301.zip']

In [51]:
# See exactly what changed
changed_versions = []

for zip_name in changed_file_zips:
    zip_path = data_folder / zip_name

    with ZipFile(zip_path) as z:
        file_bytes = z.read(changed_file)

    df = pl.read_parquet(BytesIO(file_bytes))

    df = df.with_columns(
        pl.lit(zip_name).alias("snapshot_date")
    )

    changed_versions.append(df)

pl.concat(changed_versions)

match_id,market_id,market_name,is_live,suspended,initial_fractional_value,fractional_value,choice_name,choice_source_id,winnig,change,snapshot_date
i64,i64,str,bool,bool,str,str,str,i64,bool,i64,str
12111784,1,"""full_time""",false,false,"""2/5""","""2/5""","""1""",7167106,true,0,"""20240228.zip"""
12111784,1,"""full_time""",false,false,"""7/4""","""7/4""","""2""",7167114,false,0,"""20240228.zip"""
12111784,1,"""full_time""",false,false,"""2/5""","""2/5""","""1""",7167106,true,0,"""20240229.zip"""
12111784,1,"""full_time""",false,false,"""7/4""","""7/4""","""2""",7167114,false,0,"""20240229.zip"""
12111784,1,"""full_time""",true,false,"""8/5""","""11/8""","""1""",7167106,true,0,"""20240301.zip"""
12111784,1,"""full_time""",true,false,"""11/7""","""16/10""","""2""",7167114,false,0,"""20240301.zip"""


The same odds file can contain different values on different dates, so we preserve the date of each observation in a column named snapshot_date.

In [52]:
# Convert the ZIP filename stored in snapshot_date into a proper Date value

changed_versions_df = pl.concat(changed_versions).with_columns(
    pl.col("snapshot_date")
    .str.replace(".zip", "")
    .str.to_date("%Y%m%d")
)

In [53]:
changed_versions_df.select("snapshot_date").unique()

snapshot_date
date
2024-02-28
2024-03-01
2024-02-29


## 5. Build the full odds dataset

In [54]:
# Define the folder containing the extracted daily datasets
# From this point onward, preprocessing uses Parquet files from this folder rather than reading them directly from the ZIP archives

EXTRACT_ROOT = data_folder / "extracted"

EXTRACT_ROOT

WindowsPath('d:/Learning/Daneshkar/Statistics/Stat with Python/Mini Project/Tennis Schema/tennis_data/extracted')

In [55]:
# Find all extracted odds Parquet files across the daily snapshot folders
# Verify how many files were found and that the files are accessible

extracted_odds_files = sorted(
    EXTRACT_ROOT.glob("*/raw_odds_parquet/*.parquet")
)

print("Number of files:", len(extracted_odds_files))
print("First file:", extracted_odds_files[0])
print("Exists:", extracted_odds_files[0].exists())

Number of files: 22065
First file: d:\Learning\Daneshkar\Statistics\Stat with Python\Mini Project\Tennis Schema\tennis_data\extracted\20240201\raw_odds_parquet\odds_11974053.parquet
Exists: True


In [56]:
# Read and combine all extracted odds files

odds_frames = []

for file in extracted_odds_files:
    df = pl.read_parquet(file)

    df = df.with_columns(
        # Some files have all-null values in this column,
        # so force one consistent datatype.
        pl.col("winnig").cast(pl.Boolean, strict=False),

        # Get the snapshot date from the parent date folder.
        pl.lit(file.parent.parent.name)
        .str.to_date("%Y%m%d")
        .alias("snapshot_date")
    )

    odds_frames.append(df)

odds_all = pl.concat(odds_frames, how="vertical_relaxed")

In [57]:
odds_all.shape

(60946, 12)

## 6. Clean malformed and missing values

In [58]:
odds_all.null_count()

match_id,market_id,market_name,is_live,suspended,initial_fractional_value,fractional_value,choice_name,choice_source_id,winnig,change,snapshot_date
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,3900,0,0


We now know these are daily snapshots, an early snapshot of a match may not yet know which choice eventually won, while a later snapshot may contain True/False.
So the next thing we check is whether those nulls later become known for the same match.


In [59]:
# Investigate missing winner information across different snapshots of each match
# Check whether a match has both missing and known winner values over time

winning_status = (
    odds_all
    .group_by("match_id")
    .agg(
        pl.col("winnig").is_null().any().alias("has_null"),
        pl.col("winnig").is_not_null().any().alias("has_result")
    )
)

winning_status.filter(
    pl.col("has_null") & pl.col("has_result")
).shape

(816, 3)

In [60]:
# How many matches have winnig missing in every snapshot
winning_status.filter(
    pl.col("has_null") & ~pl.col("has_result")
).shape

(198, 3)

816 matches: winnig is missing in some snapshots but known in others.  
198 matches: winnig is missing in every snapshot.

In [61]:
# Count how many rows belong to those 198 matches
always_null_matches = (
    winning_status
    .filter(
        pl.col("has_null") & ~pl.col("has_result")
    )
    .select("match_id")
)

odds_all.join(
    always_null_matches,
    on="match_id",
    how="inner"
).shape

(1316, 12)

198 matches account for 1,316 rows out of 60,946 — about 2.2% of the odds table.

In [62]:
# Check the fractional odds formatting across the entire dataset
# Valid fractional odds should follow the "number/number" format
odds_all.select(
    (
        ~pl.col("initial_fractional_value")
        .str.contains(r"^\d+/\d+$")
    ).sum().alias("bad_initial_odds"),

    (
        ~pl.col("fractional_value")
        .str.contains(r"^\d+/\d+$")
    ).sum().alias("bad_current_odds")
)

bad_initial_odds,bad_current_odds
u32,u32
133,112


In [63]:
# Inspect the bad values
odds_all.filter(
    ~pl.col("fractional_value").str.contains(r"^\d+/\d+$")
).select(
    "fractional_value"
).unique()

fractional_value
str
"""4/"""
"""1/"""
"""13/"""
"""2/"""


In [64]:
# Clean both odds columns
odds_clean = odds_all.with_columns(
    pl.when(
        pl.col("initial_fractional_value").str.contains(r"^\d+/\d+$")
    )
    .then(pl.col("initial_fractional_value"))
    .otherwise(None)
    .alias("initial_fractional_value"),

    pl.when(
        pl.col("fractional_value").str.contains(r"^\d+/\d+$")
    )
    .then(pl.col("fractional_value"))
    .otherwise(None)
    .alias("fractional_value")
)

In [65]:
odds_clean.null_count()

match_id,market_id,market_name,is_live,suspended,initial_fractional_value,fractional_value,choice_name,choice_source_id,winnig,change,snapshot_date
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,133,112,0,0,3900,0,0


## 7. Validate the cleaned dataset

In [66]:
odds_clean.is_duplicated().sum()

0

In [67]:
# Verify that each betting selection appears only once within each daily snapshot
# Similar selections on different dates are intentionally kept as separate observations

odds_clean.group_by(
    ["snapshot_date", "match_id", "market_id", "choice_source_id"]
).len().filter(
    pl.col("len") > 1
)

snapshot_date,match_id,market_id,choice_source_id,len
date,i64,i64,i64,u32


## 8. Rename columns and save the cleaned data

In [68]:
# Fixing the 'winnig' typo
odds_clean = odds_clean.rename({
    "winnig": "winning"
})

In [69]:
odds_clean.columns

['match_id',
 'market_id',
 'market_name',
 'is_live',
 'suspended',
 'initial_fractional_value',
 'fractional_value',
 'choice_name',
 'choice_source_id',
 'winning',
 'change',
 'snapshot_date']

In [70]:
# Saving the table in the processed folder
PROCESSED_ROOT = EXTRACT_ROOT.parent / "processed"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

odds_output = PROCESSED_ROOT / "odds_clean.parquet"

odds_clean.write_parquet(odds_output)

In [71]:
print(odds_output)
print(odds_output.exists())

d:\Learning\Daneshkar\Statistics\Stat with Python\Mini Project\Tennis Schema\tennis_data\processed\odds_clean.parquet
True


## 9. Final preprocessing summary

The cleaned odds dataset contains:

- 60,946 rows and 12 columns
- Daily `snapshot_date` preserved
- `winnig` standardized to Boolean and renamed to `winning`
- Malformed fractional odds converted to null rather than guessed
- Missing winner information retained
- No exact duplicate rows
- No duplicate betting selections within the same daily snapshot
- Similar observations across different snapshot dates intentionally preserved